In [9]:
"""
Author: Rylie Fabian
Date: June 18 , 2024

"""


import numpy as np
import nnfs 
import matplotlib.pyplot as plt
from nnfs.datasets import spiral_data,vertical_data,sine_data

nnfs.init()


class Layer_Dense:
    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1 = 0, weight_regularizer_l2 = 0,
                 bias_regularizer_l1 = 0, bias_regularizer_l2 = 0):
        self.weights = 0.1* np.random.randn(n_inputs,n_neurons) # random numbers for weight
        self.biases = np.zeros((1,n_neurons)) # biases to zero

        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2

    def forward(self,inputs):
        self.output = np.dot(inputs,self.weights) + self.biases
        self.inputs = inputs
    
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis= 0 , keepdims= True)

        if self.weight_regularizer_l1 > 0:
            dl1 = np.ones_like(self.weights)
            dl1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dl1

        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        if self.bias_regularizer_l1 > 0:
            dl1 = np.ones_like(self.biases)
            dl1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dl1

        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases

        

        self.dinputs = np.dot(dvalues , self.weights.T)        

class Layer_Dropout:
    def __init__(self, rate):
        self.rate = 1 - rate 

    def forward (self,inputs):
        self.inputs = inputs 
        
        self.binary_mask = np.random.binomial(1, self.rate , size = inputs.shape) / self.rate
        
        self.output = inputs * self.binary_mask

    def backward(self , dvalues):
        self.dinputs = dvalues * self.binary_mask


class Activation_ReLU: 
    
    def forward (self,inputs):
        self.output = np.maximum(0,inputs)
        self.inputs = inputs

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()

        self.dinputs[self.inputs <= 0] = 0
    
    def predictions(self,outputs):
        return outputs

class Activation_Softmax:

    def forward (self,inputs):
        exp_values = np.exp(inputs- np.max(inputs, axis= 1, keepdims= True))

        norm_values = exp_values / np.sum(exp_values, axis= 1 , keepdims= True)

        self.output = norm_values

    def backward(self,dvalues):
        # unisitialize the dinputs array 
        # just garabge values
        # makes sure the inputs are the same shape as the d-values
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            #single_output >> 2D matrix
            single_output =  single_output.reshape(-1, 1)

            #this is the full gradient of the activation function 
            self.jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            # then dot product the jacobiam matrix with the single dvalues
            # this is the chain rule 
            self.dinputs[index] = np.dot(self.jacobian_matrix, single_dvalues)

    def prediction(self, outputs):
        return np.argmax(outputs, axis= 1)

class Activation_Linear:

    # used for more regression NN
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()

    def predictions(self,outputs):
        return outputs

class Activation_Sigmoid():
    def forward(self, inputs):
        self.inputs = inputs 
        self.output = 1 / (1 + np.exp(-inputs))
    
    def backward(self, dvalues):
        self.dinputs = dvalues * ( 1 - self.output) * self.output

    def predictions(self,outputs):
        return (outputs > 0.5) * 1


In [10]:
"""
Loss functions

"""
class Loss:
    
    def calculate(self, output, y, *, include_regularization = False):
        
        sample_losses = self.forward (output , y)

        data_loss = np.mean(sample_losses)

        if not include_regularization:
            return data_loss

        return data_loss, self.regularization_loss()
    
    def regularization_loss (self):
        
        regurlarization_loss = 0

        for layer in self.remember_trainable_layers:
            # l1 regularizer for weights
            if layer.weight_regularizer_l1 > 0:
                regurlarization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

            # l2 regularizer for weights
            if layer.weight_regularizer_l2 > 0:
                regurlarization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

            # l1 regularizer for biases
            if layer.bias_regularizer_l1 > 0:
                regurlarization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

            # l2 regularizer for biases
            if layer.bias_regularizer_l2 > 0:
                regurlarization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return regurlarization_loss
    
    def remember_trainable_layers(self, trainable_layers):
        self.remember_trainable_layers = trainable_layers 
        
class Loss_BinaryCrossEntropy(Loss):
    def forward(self,y_pred,y_true):
        # if it was zero, DIVISION BY ZEROOOOOOO
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 -1e-7)

        sample_losses = - (y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))

        sample_losses = np.mean(sample_losses, axis= -1)

        return sample_losses
    
    def backward(self,dvalues,y_true):
        samples = len(dvalues)

        outputs = len(dvalues[0])

        cliped_dvalues = np.clip (dvalues, 1e-7, 1 - 1e-7)

        self.dinputs = -(y_true / cliped_dvalues - (1 - y_true) / (1 - cliped_dvalues)) / outputs

        self.dinputs = (self.dinputs / samples)

class Loss_CatergoricalCrsossentropy(Loss):
    def forward (self,y_pred, y_true):
        samples = len(y_pred)
        
        y_pred_clipped = np.clip( y_pred , 1e-7 , 1- 1e-7)

        # For sparse Labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped [range(samples),y_true]

        # For One Hot encoded Labels
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis= 1)

        negative_log_likelihoods = -np.log(correct_confidences)

        return negative_log_likelihoods
    
    def backwards (self, dvalues, y_true):
        samples = len(dvalues)

        lables = len(dvalues[0])

        if len(y_true.shape) == 1:
            y_true = np.eye(lables)[y_true]

        
        # this is the yi,j / y^ i,j
        self.dinputs = -y_true/dvalues

        # normalizing the dinputs
        self.dinputs = self.dinputs/samples

class Loss_MeanSquaredError(Loss):
    def forward(self, y_pred, y_true):
        sample_loss = np.mean((y_true - y_pred) ** 2, axis=-1)

        return sample_loss
    
    def backward(self,dvalues , y_true):
        samples = len(dvalues)

        outputs = len(dvalues[0])

        self.dinputs = -2 * (y_true - dvalues) / outputs

        self.dinputs = self.dinputs / samples

class Loss_MeanAbsoluteError(Loss):

    def forward(self, y_pred, y_true):
        sample_loss = np.mean(np.abs(y_true - y_pred))

        return sample_loss
    
    def backward(self, dinputs, y_true):
        samples = len(dinputs)

        outputs = len(dinputs[0])

        # The sign function returns -1 if x < 0, 0 if x==0, 1 if x > 0. nan is returned for nan inputs.

        self.dinputs = np.sign(y_true - dinputs) / outputs

        self.dinputs = self.dinputs / outputs



In [ ]:
"""
Activation Funciton

"""
class Activation_Softmax_Loss_CategoricalCrossentropy(): 
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CatergoricalCrsossentropy()

    def forward (self,inputs, y_true):
        self.activation.forward(inputs = inputs)
        self.output = self.activation.output 

        return self.loss.calculate(self.output , y_true)
    
    def backward(self,dvalues , y_true):
        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true,axis= 1)

        self.dinputs = dvalues.copy()

        # the gradient calculated 
        self.dinputs[range(samples), y_true] -= 1

        self.dinputs = self.dinputs / samples
        

class Optimizer_SGD:
    def __init__(self, learning_rate = 1.0, decay = 0, momentum = 0):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay 
        self.iterations = 0 
        self.momentum = momentum

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))
    
    def update_params(self, layer):
        if self.momentum:
            
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)

                layer.bias_momentums = np.zeros_like(layer.biases)

            weight_updates = \
                self.momentum * layer.weight_momentums - \
                self.current_learning_rate * layer.dweights
            layer.weight_momentums = weight_updates

            bias_updates = \
                self.momentum * layer.bias_momentums - \
                self.current_learning_rate * layer.dbiases
            layer.bias_momentums = bias_updates

        else:
            weight_updates = -self.current_learning_rate * layer.dweights
            bias_updates = -self.current_learning_rate * layer.dbiases
            
        layer.weights += weight_updates
        layer.biases += bias_updates



    def post_update_params(self):
        self.iterations += 1


class Optimizer_Adagrad:
    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay 
        self.iterations = 0 
        self.epsilon = epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))
    
    def update_params(self, layer):
    
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # cache += parm_gradiant ^ 2
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache += layer.dbiases ** 2

        # param updates = learning_rate * gradient / (sqrt(cache) + epsilon)
        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)


    def post_update_params(self):
        self.iterations += 1
    
class Optimizer_RMSprop():
    def __init__(self, learning_rate = 0.001, decay = 0, epsilon = 1e-7, rho = 0.9):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay 
        self.iterations = 0 
        self.epsilon = epsilon
        self.rho = rho

    def pre_update_params(self):
        if self.decay :
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_cache = self.rho * layer.weight_cache + (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache = self.rho * layer.bias_cache + (1 - self.rho) * layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1 

class Optimizer_Adam():
    def __init__(self, learning_rate = 0.001, decay = 0, epsilon = 1e-7, beta_1 = 0.9, beta_2 = 0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay 
        self.iterations = 0 
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay :
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations +1))

        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        layer.weight_cache = self.beta_2 * layer.weight_cache + (1- self.beta_2) * layer.dweights ** 2
        layer.bias_cache = self.beta_2 * layer.bias_cache + (1- self.beta_2) * layer.dbiases ** 2


        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations +1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations +1))

        layer.weights += -self.current_learning_rate * weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases += -self.current_learning_rate * bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1 
    


In [12]:
class Model:
    def __init__(self):
        self.layers = []

    def add (self,layer):
        self.layers.append(layer)
    
    def set(self, *,loss ,optimizer , accuracy):
        self.loss = loss
        self.optimizer = optimizer
        self.accuracy = accuracy



        
    def finalize(self):
        self.input_layer = Layer_Input()

        layer_count = len(self.layers)

        self.trainable_layers = []

        for i in range(layer_count):
            if i == 0: # first layer
                self.layers[i].prev = self.input_layer
                self.layers[i].next = self.layers[i+1]

            elif i < layer_count - 1: # hidden layers
                self.layers[i].prev = self.layers[i-1]
                self.layers[i].next = self.layers[i+1]
                
            else: #Last layer
                self.layers[i].prev = self.layers[i-1]
                self.layers[i].next = self.loss
                self.output_layer_activation = self.layers[i]

            if hasattr(self.layers[i], "weights"):
                self.trainable_layers.append(self.layers[i])

        self.loss.remember_trainable_layers(self.trainable_layers)
                
    def forward(self , x):
        self.input_layer.forward(x)

        for layer in self.layers:
            layer.forward(layer.prev.output)
        
        return layer.output

    def train(self,x , y , *, epochs = 1, print_every = 1, validation_data = None):
        self.accuracy.init(y)


        if validation_data is not None:
            # For better readability
            X_val, y_val = validation_data
            # Perform the forward pass
            output = self.forward(X_val)
            # Calculate the loss
            loss = self.loss.calculate(output, y_val)
            # Get predictions and calculate an accuracy
            predictions = self.output_layer_activation.predictions(
            output)
            accuracy = self.accuracy.calculate(predictions, y_val)
            # Print a summary
            print(f'validation, ' +
            f'acc: {accuracy:.3f}, ' +
            f'loss: {loss:.3f}')



        for epoch in range (1,epochs + 1):
            output = self.forward(x)

            data_loss, regularization_loss = \
                self.loss.calculate(output,y,include_regularization = True)
            loss = data_loss + regularization_loss

            predictions = self.output_layer_activation.predictions(output)
            accuracy = self.accuracy.calculate(predictions, y)

            self.backward(output, y)

            self.optimizer.pre_update_params()
            for layer in self.trainable_layers:
                self.optimizer.update_params(layer)
            self.optimizer.post_update_params()

            if not epoch % print_every:
                print(f'epoch: {epoch}, ' +
                f'acc: {accuracy:.3f}, ' +
                f'loss: {loss:.3f} (' +
                f'data_loss: {data_loss:.3f}, ' +
                f'reg_loss: {regularization_loss:.3f}), ' +
                f'lr: {self.optimizer.current_learning_rate}')

    def backward(self,output,y):
        self.loss.backward(output,y)
        
        for layer in reversed(self.layers):
            layer.backward(layer.next.dinputs)



class Accuracy:
    def calculate(self, predicitons, y):
        comparisons = self.compare(predicitons, y)

        accuracy = np.mean(comparisons)

        return accuracy
    
class Accuracy_Regression(Accuracy):
    def __init__(self):
        self.percision = None
    
    def init(self,y,reinit = False):
        if self.percision is None or reinit:
            self.percision = np.std(y) / 250

    def compare(self, predictions , y):
        return np.absolute(predictions - y) < self.percision


class Layer_Input:
    def forward(self,inputs):
        self.output = inputs    


class Accuracy_Categorical(Accuracy):

    def __init__(self, *, binary = False):
        self.binary = binary

    def init(self,y):
        pass
    
    def compare(self,predictions, y):
        
        if not self.binary and len(y.shape) == 2:
            
            y = np.argmax(y, axis=1)
        
        return predictions == y
        
        

In [13]:

"""
Testing the model
"""

# Create train and test dataset
X, y = spiral_data(samples=100, classes=2)
X_test, y_test = spiral_data(samples=100, classes=2)
# Reshape labels to be a list of lists
# Inner list contains one output (either 0 or 1)
# per each output neuron, 1 in this case
y = y.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)
# Instantiate the model
model = Model()
# Add layers
model.add(Layer_Dense(2, 64, weight_regularizer_l2=5e-4,
 bias_regularizer_l2=5e-4))
model.add(Activation_ReLU())
model.add(Layer_Dense(64, 1))
model.add(Activation_Sigmoid())

# Set loss, optimizer and accuracy objects
model.set(
 loss=Loss_BinaryCrossEntropy(),
 optimizer=Optimizer_Adam(decay=5e-7),
 accuracy=Accuracy_Categorical(binary=True)
)
# Finalize the model
model.finalize()
# Train the model
model.train(X, y, validation_data=(X_test, y_test),
 epochs=10000, print_every=100)

validation, acc: 0.500, loss: 0.691
epoch: 100, acc: 0.610, loss: 0.672 (data_loss: 0.671, reg_loss: 0.001), lr: 0.0009999505024501287
epoch: 200, acc: 0.610, loss: 0.667 (data_loss: 0.665, reg_loss: 0.002), lr: 0.0009999005098992651
epoch: 300, acc: 0.630, loss: 0.661 (data_loss: 0.659, reg_loss: 0.003), lr: 0.000999850522346909
epoch: 400, acc: 0.645, loss: 0.654 (data_loss: 0.650, reg_loss: 0.004), lr: 0.0009998005397923115
epoch: 500, acc: 0.650, loss: 0.645 (data_loss: 0.639, reg_loss: 0.006), lr: 0.0009997505622347225
epoch: 600, acc: 0.645, loss: 0.635 (data_loss: 0.628, reg_loss: 0.008), lr: 0.0009997005896733929
epoch: 700, acc: 0.675, loss: 0.626 (data_loss: 0.616, reg_loss: 0.010), lr: 0.0009996506221075735
epoch: 800, acc: 0.720, loss: 0.616 (data_loss: 0.604, reg_loss: 0.012), lr: 0.000999600659536515
epoch: 900, acc: 0.750, loss: 0.606 (data_loss: 0.592, reg_loss: 0.014), lr: 0.0009995507019594694
epoch: 1000, acc: 0.760, loss: 0.597 (data_loss: 0.581, reg_loss: 0.016), l